# Stage 3: Solving Severe Target Class Imbalance (84.8% vs. 15.2%)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Standardized Data Loading:** Load `framingham_engineered_selected.csv` and apply Z-score standardization (`StandardScaler`).
2. **Mathematical Implementation of Five Resampling Algorithms:** Implement `Standard SMOTE`, `ADASYN`, `Hybrid SMOTE + Tomek Links`, `Borderline-SMOTE`, and `Hybrid SMOTE + Edited Nearest Neighbours (ENN)`.
3. **Five-Algorithm Imbalance Resolution Benchmark:** Compare all five methods against the unprocessed imbalanced baseline under 5-Fold Stratified Cross-Validation.
4. **Feature Space Visualization & Balanced Dataset Export:** Plot the 2D PCA boundary separation (`fig7`) and export `framingham_balanced.csv` (`6,147` samples).

### 1. Environment Setup and Data Standardization
Load the 20 selected features from Stage 2 and standardize them using the winning `StandardScaler`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

df_fe = pd.read_csv('framingham_engineered_selected.csv')
feature_names = list(df_fe.drop('TenYearCHD', axis=1).columns)
X_raw = df_fe.drop('TenYearCHD', axis=1).values
y = df_fe['TenYearCHD'].values

# Apply winning Z-score standardization (StandardScaler)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_raw)

print(f"Original Imbalanced Shape: {X_sc.shape}")
print(f"Class 0 (Healthy): {np.sum(y==0)} ({np.mean(y==0)*100:.2f}%)")
print(f"Class 1 (CHD):     {np.sum(y==1)} ({np.mean(y==1)*100:.2f}%)")

### 2. Mathematical Implementation of Five Synthetic Resampling & Boundary-Cleaning Algorithms
Define vectorized nearest-neighbor implementations for `SMOTE`, `ADASYN`, `SMOTE + Tomek Links`, `Borderline-SMOTE`, and `Hybrid SMOTE + ENN`.

In [ ]:
# 1. Standard SMOTE (Chawla et al., 2002)
def smote_resample(X_in, y_in, k_neighbors=5, random_state=42):
    rng = np.random.RandomState(random_state)
    X_maj = X_in[y_in == 0]
    X_min = X_in[y_in == 1]
    n_syn = len(X_maj) - len(X_min)
    nn = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(X_min)
    _, indices = nn.kneighbors(X_min)
    syn_samples = []
    for _ in range(n_syn):
        i = rng.randint(0, len(X_min))
        neighbor_idx = indices[i, rng.randint(1, k_neighbors + 1)]
        lam = rng.uniform(0, 1)
        syn = X_min[i] + lam * (X_min[neighbor_idx] - X_min[i])
        syn_samples.append(syn)
    X_res = np.vstack([X_in, np.array(syn_samples)])
    y_res = np.hstack([y_in, np.ones(n_syn, dtype=y_in.dtype)])
    return X_res, y_res

# 2. ADASYN (Adaptive Synthetic Sampling - He et al., 2008)
def adasyn_resample(X_in, y_in, k_neighbors=5, random_state=42):
    rng = np.random.RandomState(random_state)
    X_maj = X_in[y_in == 0]
    X_min = X_in[y_in == 1]
    n_syn = len(X_maj) - len(X_min)
    nn_all = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(X_in)
    _, ind_all = nn_all.kneighbors(X_min)
    ratios = np.array([np.sum(y_in[ind_all[i, 1:]] == 0) / k_neighbors for i in range(len(X_min))])
    probs = ratios / ratios.sum() if ratios.sum() > 0 else np.ones(len(X_min)) / len(X_min)
    nn_min = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(X_min)
    _, ind_min = nn_min.kneighbors(X_min)
    syn_samples = []
    chosen_indices = rng.choice(len(X_min), size=n_syn, p=probs)
    for i in chosen_indices:
        neighbor_idx = ind_min[i, rng.randint(1, k_neighbors + 1)]
        lam = rng.uniform(0, 1)
        syn = X_min[i] + lam * (X_min[neighbor_idx] - X_min[i])
        syn_samples.append(syn)
    X_res = np.vstack([X_in, np.array(syn_samples)])
    y_res = np.hstack([y_in, np.ones(n_syn, dtype=y_in.dtype)])
    return X_res, y_res

# 3. Hybrid SMOTE + Tomek Links (Batista et al., 2004)
def smote_tomek_resample(X_in, y_in, k_neighbors=5, random_state=42):
    X_sm, y_sm = smote_resample(X_in, y_in, k_neighbors=k_neighbors, random_state=random_state)
    nn_1 = NearestNeighbors(n_neighbors=2).fit(X_sm)
    _, ind_1 = nn_1.kneighbors(X_sm)
    nearest = ind_1[:, 1]
    tomek_mask = np.zeros(len(X_sm), dtype=bool)
    for i in range(len(X_sm)):
        j = nearest[i]
        if nearest[j] == i and y_sm[i] != y_sm[j]:
            tomek_mask[i] = True
            tomek_mask[j] = True
    return X_sm[~tomek_mask], y_sm[~tomek_mask]

# 4. Borderline-SMOTE (Danger-Zone Interpolation - Han et al., 2005)
def borderline_smote_resample(X_in, y_in, k_neighbors=5, m_neighbors=10, random_state=42):
    rng = np.random.RandomState(random_state)
    X_maj = X_in[y_in == 0]
    X_min = X_in[y_in == 1]
    n_syn = len(X_maj) - len(X_min)
    nn_all = NearestNeighbors(n_neighbors=m_neighbors + 1).fit(X_in)
    _, ind_all = nn_all.kneighbors(X_min)
    danger_mask = [
        (m_neighbors / 2.0 <= np.sum(y_in[ind_all[i, 1:]] == 0) < m_neighbors)
        for i in range(len(X_min))
    ]
    X_danger = X_min[np.array(danger_mask)]
    if len(X_danger) < 5:
        X_danger = X_min
    nn_min = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(X_min)
    _, ind_min = nn_min.kneighbors(X_danger)
    syn_samples = []
    for _ in range(n_syn):
        i = rng.randint(0, len(X_danger))
        neighbor_idx = ind_min[i, rng.randint(1, k_neighbors + 1)]
        lam = rng.uniform(0, 1)
        syn = X_danger[i] + lam * (X_min[neighbor_idx] - X_danger[i])
        syn_samples.append(syn)
    X_res = np.vstack([X_in, np.array(syn_samples)])
    y_res = np.hstack([y_in, np.ones(n_syn, dtype=y_in.dtype)])
    return X_res, y_res

# 5. Hybrid SMOTE + ENN (Edited Nearest Neighbours - Winning Method)
def smote_enn_resample(X_in, y_in, k_neighbors=5, enn_k=3, random_state=42):
    X_sm, y_sm = smote_resample(X_in, y_in, k_neighbors=k_neighbors, random_state=random_state)
    nn_e = NearestNeighbors(n_neighbors=enn_k + 1).fit(X_sm)
    _, ind_e = nn_e.kneighbors(X_sm)
    keep_mask = np.ones(len(X_sm), dtype=bool)
    for i in range(len(X_sm)):
        neigh_labels = y_sm[ind_e[i, 1:]]
        maj_vote = 1 if np.sum(neigh_labels == 1) > (enn_k / 2.0) else 0
        if y_sm[i] != maj_vote:
            keep_mask[i] = False
    return X_sm[keep_mask], y_sm[keep_mask]

### 3. Benchmark All Five Imbalance Resolution Algorithms
Evaluate each resampling algorithm using 5-Fold Stratified Cross-Validation across Recall, Precision, F1-Score, ROC-AUC, and Accuracy.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_eval = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)

imbalance_methods = [
    ('0. Imbalanced Baseline (84.8% vs 15.2%)', None),
    ('1. Standard SMOTE (k=5)', smote_resample),
    ('2. ADASYN (Adaptive Synthetic)', adasyn_resample),
    ('3. Hybrid SMOTE + Tomek Links', smote_tomek_resample),
    ('4. Borderline-SMOTE (Danger Zone)', borderline_smote_resample),
    ('5. Hybrid SMOTE + ENN (Winner)', smote_enn_resample)
]

bench_rows = []
for name, fn in imbalance_methods:
    Xr, yr = (X_sc, y) if fn is None else fn(X_sc, y)
    cv_res = cross_validate(
        clf_eval, Xr, yr, cv=skf,
        scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs=-1
    )
    bench_rows.append({
        'Algorithm': name,
        'Class_0 / Class_1': f"{np.sum(yr==0)} / {np.sum(yr==1)}",
        'Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'F1-Score (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'ROC-AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2)
    })

imb_df = pd.DataFrame(bench_rows)
print("=== Class Imbalance Resolution Algorithms Benchmark (5-Fold Stratified CV) ===")
print(imb_df.to_string(index=False))

### 4. Export Balanced Dataset & Plot Boundary Separation (Figure 7)
Save the winning `Hybrid SMOTE + ENN` balanced dataset to `framingham_balanced.csv` and visualize the cleaned feature space.

In [ ]:
# Export the winning Hybrid SMOTE + ENN dataset
X_enn, y_enn = smote_enn_resample(X_sc, y)
df_balanced = pd.DataFrame(X_enn, columns=feature_names)
df_balanced['TenYearCHD'] = y_enn
df_balanced.to_csv('framingham_balanced.csv', index=False)
print(f"\n[Successfully saved winning balanced dataset to: framingham_balanced.csv with shape {df_balanced.shape}]")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

pca_2d = PCA(n_components=2, random_state=42)
X_2d_orig = pca_2d.fit_transform(X_sc)
X_2d_enn = pca_2d.transform(X_enn)

axes[0].scatter(X_2d_enn[y_enn==0, 0], X_2d_enn[y_enn==0, 1], c='#0ea5e9', alpha=0.35, s=15, label='Healthy Class 0 (ENN Cleaned: 2,594)')
axes[0].scatter(X_2d_enn[y_enn==1, 0], X_2d_enn[y_enn==1, 1], c='#e11d48', alpha=0.35, s=15, label='CHD Class 1 (SMOTE+ENN: 3,553)')
axes[0].set_title('A. Feature Space After Hybrid SMOTE + ENN (Clean Boundary Separation)', fontweight='bold')
axes[0].set_xlabel('Principal Component 1 (PC1)')
axes[0].set_ylabel('Principal Component 2 (PC2)')
axes[0].legend(loc='upper right', frameon=True)
axes[0].set_xlim(-5, 8)
axes[0].set_ylim(-5, 8)

alg_labels = ['0. Imbalanced\n(85% vs 15%)', '1. Standard\nSMOTE', '2. ADASYN', '3. SMOTE +\nTomek Links', '4. Borderline\nSMOTE', '5. Hybrid\nSMOTE + ENN']
rec_list  = imb_df['Recall (%)'].values
prec_list = imb_df['Precision (%)'].values
f1_list   = imb_df['F1-Score (%)'].values
auc_list  = imb_df['ROC-AUC (%)'].values

x_p = np.arange(len(alg_labels))
w = 0.20
axes[1].bar(x_p - 1.5*w, rec_list, width=w, label='Recall (%)', color='#e11d48', edgecolor='black')
axes[1].bar(x_p - 0.5*w, prec_list, width=w, label='Precision (%)', color='#8b5cf6', edgecolor='black')
axes[1].bar(x_p + 0.5*w, f1_list, width=w, label='F1-Score (%)', color='#f59e0b', edgecolor='black')
axes[1].bar(x_p + 1.5*w, auc_list, width=w, label='ROC-AUC (%)', color='#10b981', edgecolor='black')

for i in range(len(alg_labels)):
    axes[1].text(x_p[i] - 1.5*w, rec_list[i] + 1.2, f"{rec_list[i]:.0f}%", ha='center', fontsize=7.5, fontweight='bold', color='#be123c')
    axes[1].text(x_p[i] + 0.5*w, f1_list[i] + 1.2, f"{f1_list[i]:.0f}%", ha='center', fontsize=7.5, fontweight='bold', color='#b45309')

axes[1].set_xticks(x_p)
axes[1].set_xticklabels(alg_labels, fontweight='bold', fontsize=8.5)
axes[1].set_ylim(0, 106)
axes[1].set_ylabel('Score (%)')
axes[1].set_title('B. Imbalance Algorithms Benchmark (Recall, Precision, F1, ROC-AUC)', fontweight='bold')
axes[1].legend(loc='lower right', ncol=2, frameon=True)

plt.tight_layout()
plt.savefig('fig7_imbalance_handling_benchmark.png', bbox_inches='tight')
plt.show()